In [48]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)

In [ ]:
# 데이터 확인하기 2025.11.21
# 이상인 컬럼 제거 후 RandomForest 기본 모델 돌리기
import pandas as pd
import numpy  as np
from scipy.special import logit

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler # 데이터 전처리용
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix, recall_score, precision_score
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

import matplotlib.pyplot as plt
import seaborn as sns

import importlib
from utils import preprocessing

# 모듈 reload
importlib.reload(preprocessing)
# importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns
from utils.user_utils    import get_model_train_eval
from utils.model_utils   import save_model, load_model

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

# train = pd.read_csv("../data/train.csv")
# test  = pd.read_csv("../data/test.csv")

In [50]:
# ======================================================
# 1) ThresholdModel 감싸진 pkl에서 base_model 추출 함수
# ======================================================
def extract_base(model):
    """
    ThresholdModel 또는 ThresholdModel_rf 의 pkl을 로드하면
    model.base_model 형태로 원본 모델을 꺼낼 수 있음.
    Threshold 미적용 모델일 경우 그대로 반환.
    """
    if hasattr(model, "base_model"):
        return model.base_model
    return model

In [51]:
class ThresholdModel:
    def __init__(self, base_model, threshold):
        self.base_model = base_model
        self.threshold = threshold

    def fit(self, X, y):
        return self

    def predict(self, X):
        proba = self.base_model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.base_model.predict_proba(X)


class ThresholdModel_rf:
    def __init__(self, base_model, threshold):
        self.base_model = base_model
        self.threshold = threshold

    def fit(self, X, y):
        return self

    def predict(self, X):
        proba = self.base_model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.base_model.predict_proba(X)


In [52]:
train, test = load_data()

X_features, y_labels = split_features_target(train)
X_test = test.drop(columns=["ID"], errors="ignore")

# var3 처리
X_features["var3"] = X_features["var3"].replace(-999999, 2)
X_test["var3"] = X_test["var3"].replace(-999999, 2)

X_train, X_val, y_train, y_val = data_split(X_features, y_labels)

In [53]:
# train/val 분리
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [54]:
xgb_thr = load_model("../models/XGB_100_HP_max5_est250_lr0.12_thr_0.15.pkl")
lgbm_thr = load_model("../models/LGBM_100_HP_lr0.5_max6_min17_est350_num16_class1vs4_thr_0.44.pkl")
rf_thr = load_model("../models/RF_100_HP_max7_est350_classWeight1vs4_thr_0.17.pkl")

✓ 모델 로드 완료: ../models\../models/XGB_100_HP_max5_est250_lr0.12_thr_0.15.pkl
  모델 타입: ThresholdModel
✓ 모델 로드 완료: ../models\../models/LGBM_100_HP_lr0.5_max6_min17_est350_num16_class1vs4_thr_0.44.pkl
  모델 타입: ThresholdModel
✓ 모델 로드 완료: ../models\../models/RF_100_HP_max7_est350_classWeight1vs4_thr_0.17.pkl
  모델 타입: ThresholdModel_rf


In [55]:
xgb = extract_base(xgb_thr)
lgbm = extract_base(lgbm_thr)
rf   = extract_base(rf_thr)

In [56]:
# -------------------------------------------------------------------
# 3) Logit 변환 함수 정의
# -------------------------------------------------------------------
def to_logit(p):
    eps = 1e-15
    return np.log(p + eps) - np.log(1 - p + eps)

In [57]:
# ============================================================
# 1) Base 모델 확률 예측
# ============================================================

print("\n[3] base 모델 확률 예측 및 logit 변환...")

xgb_proba = xgb.predict_proba(X_val_scaled)[:, 1]
lgbm_proba = lgbm.predict_proba(X_val)[:, 1]
rf_proba   = rf.predict_proba(X_val)[:, 1]

# logit 변환
eps = 1e-6
xgb_logit  = logit(np.clip(xgb_proba, eps, 1-eps))
lgbm_logit = logit(np.clip(lgbm_proba, eps, 1-eps))
rf_logit   = logit(np.clip(rf_proba, eps, 1-eps))

# stacking input matrix
stack_val = np.vstack([xgb_logit, lgbm_logit, rf_logit]).T


[3] base 모델 확률 예측 및 logit 변환...


In [58]:
# ======================================
# 2) 메타 모델 정의 및 학습
# ======================================
# meta = LogisticRegression(max_iter=2000, class_weight="balanced")
meta = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=5,   # 희귀 타겟 보정
    eval_metric='logloss'
)
meta.fit(stack_val, y_val)

# 메타 모델 예측
stack_proba = meta.predict_proba(stack_val)[:, 1]
stack_pred_default = (stack_proba >= 0.5).astype(int)

In [ ]:
print("\n===== Stacking 기본 성능 =====")
auc = roc_auc_score(y_val, stack_proba)
acc = accuracy_score(y_val, stack_pred_default)
f1  = f1_score(y_val, stack_pred_default)

print(f"AUC : {auc:.4f}")
print(f"ACC : {acc:.4f}")
print(f"F1  : {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, stack_pred_default))

print("\nClassification Report:")
print(classification_report(y_val, stack_pred_default))

# # 메타 기여도 출력
# print("\n[기여도] 메타 모델 Coefficients")
# for name, c in zip(["XGB", "LGBM", "RF"], meta.coef_[0]):
#     print(f"{name}: {c:.5f}")

# ===== Stacking 기본 성능 =====
# AUC : 0.8486
# ACC : 0.7654
# F1  : 0.2068

# Confusion Matrix:
# [[11172  3430]
#  [  137   465]]


===== Stacking 기본 성능 =====
AUC : 0.8900
ACC : 0.9253
F1  : 0.3098

Confusion Matrix:
[[13813   789]
 [  347   255]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.95      0.96     14602
           1       0.24      0.42      0.31       602

    accuracy                           0.93     15204
   macro avg       0.61      0.68      0.64     15204
weighted avg       0.95      0.93      0.93     15204



In [60]:
# ------------------------------------------------
# 9) Threshold 최적화
# ------------------------------------------------
print("\n[5] Threshold 최적화 중...")

best_f1 = 0
best_thr = 0

for thr in np.arange(0.01, 0.50, 0.01):
    pred_thr = (stack_proba >= thr).astype(int)
    f1_thr = f1_score(y_val, pred_thr)
    if f1_thr > best_f1:
        best_f1 = f1_thr
        best_thr = thr

print(f"\n>>> 최적 Threshold = {best_thr:.2f}, Best F1 = {best_f1:.4f}")


[5] Threshold 최적화 중...

>>> 최적 Threshold = 0.47, Best F1 = 0.3193


In [61]:
# ------------------------------------------------
# 10) 최종 모델 클래스 정의 및 저장
# ------------------------------------------------
class FinalStackingModel:
    """
    meta + threshold 두 개만 저장.
    base 모델은 stacking 단계에서 이미 확률값이 input으로 활용됨.
    """
    def __init__(self, meta_model, threshold):
        self.meta = meta_model
        self.threshold = threshold

    def predict_proba(self, X):
        return self.meta.predict_proba(X)

    def predict(self, X):
        proba = self.meta.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

In [ ]:
# ------------------------------------------------
# 11) 최종 Threshold 적용 성능 출력
# ------------------------------------------------
print("\n===== Stacking Threshold 적용 성능 =====")

stack_pred_best = (stack_proba >= best_thr).astype(int)

auc_best = roc_auc_score(y_val, stack_proba)   # AUC는 threshold 상관없음
acc_best = accuracy_score(y_val, stack_pred_best)
f1_best  = f1_score(y_val, stack_pred_best)
cm_best  = confusion_matrix(y_val, stack_pred_best)

print(f"AUC (same): {auc_best:.4f}")
print(f"ACC : {acc_best:.4f}")
print(f"F1  : {f1_best:.4f}")

print("\nConfusion Matrix:")
print(cm_best)

print("\nClassification Report:")
print(classification_report(y_val, stack_pred_best))

print(f"\n>>> 적용 Threshold : {best_thr:.2f}")

class FinalStackingXGB:
    def __init__(self, meta_model, threshold):
        self.meta = meta_model
        self.threshold = threshold

    def predict_proba(self, X):
        return self.meta.predict_proba(X)

    def predict(self, X):
        proba = self.meta.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

final_model = FinalStackingXGB(meta, best_thr)

save_model(final_model, f"Stacking_100_XGBmeta_thr{best_thr:.2f}.pkl")

print("\n===== Meta XGB Stacking Pipeline 완료 =====")

# ===== Stacking Threshold 적용 성능 =====
# AUC (same): 0.8900
# ACC : 0.9215
# F1  : 0.3193

# Confusion Matrix:
# [[13730   872]
#  [  322   280]]

# Classification Report:
#               precision    recall  f1-score   support

#            0       0.98      0.94      0.96     14602
#            1       0.24      0.47      0.32       602

#     accuracy                           0.92     15204
#    macro avg       0.61      0.70      0.64     15204
# weighted avg       0.95      0.92      0.93     15204


# >>> 적용 Threshold : 0.47
# ✓ 모델 저장 완료: ../models\Stacking_100_XGBmeta_thr0.47.pkl.pkl
#   파일 크기: 0.22 MB

# ===== Meta XGB Stacking Pipeline 완료 =====


===== Stacking Threshold 적용 성능 =====
AUC (same): 0.8900
ACC : 0.9215
F1  : 0.3193

Confusion Matrix:
[[13730   872]
 [  322   280]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.94      0.96     14602
           1       0.24      0.47      0.32       602

    accuracy                           0.92     15204
   macro avg       0.61      0.70      0.64     15204
weighted avg       0.95      0.92      0.93     15204


>>> 적용 Threshold : 0.47
✓ 모델 저장 완료: ../models\Stacking_100_XGBmeta_thr0.47.pkl.pkl
  파일 크기: 0.22 MB

===== Meta XGB Stacking Pipeline 완료 =====


In [63]:
print("\n[6] 최종 모델 저장 중...")

final_model = FinalStackingModel(meta, best_thr)
save_model(final_model, f"StackingModel_100_final_thr{best_thr:.2f}.pkl")

print("\n===== 최종 Stacking Pipeline 완료 =====")


[6] 최종 모델 저장 중...
✓ 모델 저장 완료: ../models\StackingModel_100_final_thr0.47.pkl.pkl
  파일 크기: 0.22 MB

===== 최종 Stacking Pipeline 완료 =====


In [64]:
# -------------------------------------------------------------------
# 7) 메타 모델 기여도 출력
# -------------------------------------------------------------------
print("\n[4] 메타 모델 기여도 (Coefficients)")
coef = meta.coef_[0]
for name, c in zip(["XGB_logit", "LGBM_logit", "RF_logit"], coef):
    print(f"{name}: {c:.4f}")

print("\n===== 스태킹 앙상블 완료 =====")

save_model(meta, "StackingModel_log1p_XGB_LGBM_RF_20251124")
print("\n✓ 스태킹 메타모델 저장 완료!")


[4] 메타 모델 기여도 (Coefficients)


AttributeError: Coefficients are not defined for Booster type None